# Time Series I: Stationarity, Random Walks & Unit Roots. Exercises
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C., *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 6 and 8

Ten hands-on problems on stationarity, random walks, the ADF test, forecasting a unit-root series and spurious regression. Work through them in order; the data cell below loads the 10-year Treasury yield (live if possible, seeded fallback otherwise) and the BTC / S&P 500 pair. Solutions are in the separate solutions notebook. Do not peek before trying.

---

## Step 0: Data Setup (run this first)

In [1]:
!pip install "statsmodels>=0.15" yfinance --quiet

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import logging
# a download that cannot reach the internet must not leave its own error text
# in the saved output; the notebook falls back to the seeded series instead.
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# ---- Dataset 1: US 10-year Treasury yield, monthly since 2000, as in the lecture notebook ----
def load_yield():
    try:
        import yfinance as yf
        raw = yf.download('^TNX', start='2000-01-01', progress=False)
        close = raw['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close = close.dropna()
        if len(close) < 1000:
            raise ValueError('download too short or empty')
        if close.median() > 20:      # arrived on the 10x CBOE index scale
            close = close / 10.0
        yy = close.resample('ME').last().dropna()
        yy.name = 'yield10y'
        return yy, 'LIVE Yahoo Finance (^TNX)'
    except Exception as e:
        rng = np.random.default_rng(9)      # seed chosen to match the lecture fallback
        path = np.cumsum(rng.normal(0, 0.25, 318))
        path = path - path[-1] + 4.30
        idx = pd.date_range('2000-01-31', periods=318, freq='ME')
        return pd.Series(path, index=idx, name='yield10y'), 'SEEDED FALLBACK (seed 9)'

# ---- Dataset 2: BTC and S&P 500 levels, monthly since 2015, as in the lecture notebook ----
def load_pair():
    try:
        import yfinance as yf
        raw = yf.download(['BTC-USD', '^GSPC'], start='2015-01-01', progress=False)['Close']
        raw = raw.dropna()
        if len(raw) < 500:
            raise ValueError('download too short or empty')
        pr = raw.resample('ME').last().dropna()
        pr.columns = ['btc', 'spx'] if pr.columns[0] == 'BTC-USD' else ['spx', 'btc']
        return pr[['btc', 'spx']], 'LIVE Yahoo Finance (BTC-USD, ^GSPC)'
    except Exception as e:
        rng = np.random.default_rng(236)    # two INDEPENDENT walks, seed as in the lecture
        m = 138
        btc = 400 * np.exp(np.cumsum(rng.normal(0.04, 0.20, m)))
        spx = 2000 * np.exp(np.cumsum(rng.normal(0.008, 0.045, m)))
        idx = pd.date_range('2015-01-31', periods=m, freq='ME')
        return pd.DataFrame({'btc': btc, 'spx': spx}, index=idx), 'SEEDED FALLBACK (seed 236)'

y, y_source = load_yield()
dy = y.diff().dropna()
pair, pair_source = load_pair()

print(f"yield : {y_source}, n = {len(y)}, today {y.iloc[-1]:.2f} percent")
print(f"pair  : {pair_source}, n = {len(pair)}")

yield : SEEDED FALLBACK (seed 9), n = 318, today 4.30 percent
pair  : SEEDED FALLBACK (seed 236), n = 138


## Exercise 1: Look First, Test Later
(a) Plot the yield level `y` and its monthly changes `dy` in two panels.
(b) State the three conditions of weak stationarity.
(c) Classify each of the two series by eye against the three conditions, and explain why an
eyeball verdict is not enough.

In [2]:
# your code here


## Exercise 2: The Difference Operator by Hand
(a) Compute the first difference of the last four observations of `y` by hand from the levels
(print the levels, then subtract on paper or in a comment).
(b) Verify with `y.diff()` that your hand-made numbers agree.
(c) One sentence: what does Delta y measure economically for a yield series, and why does
differencing cost one observation?

In [3]:
# your code here


## Exercise 3: Reading the ACF
(a) Plot the ACF of the level and of the changes (24 lags each).
(b) Report the lag-1 autocorrelation of both series, and the level's ACF at lags 12 and 24.
(c) A geometric decay at a rate of 0.98 per lag would stand at 0.98 to the power 12 after
a year. Compute that value and compare it with your measured lag-12 value. What is the
fingerprint you should remember?

In [4]:
# your code here


## Exercise 4: Simulate Random Walks and Check the Variance Law
(a) Simulate 2000 random-walk paths of length 100 with N(0, 1) shocks (one seeded generator).
(b) Compute the cross-path variance of the walk at t = 10, 25, 50 and 100 and compare with the
theoretical Var(y_t) = t sigma-squared.
(c) Why does this growing variance violate stationarity, and what does it imply for the
uncertainty of long-horizon forecasts?

In [5]:
# your code here


## Exercise 5: The ADF Test on the Level
(a) Run `adfuller` on the yield level with a constant, `autolag='BIC'` and
`result_object=True`, which lets you read the results by name. Report the test
statistic, the p-value and the three critical values.
(b) State H0 and H1 of the test and make the 5 percent decision.
(c) A colleague compares your statistic with -1.645 and announces mean reversion. What exactly
is the mistake?

In [6]:
# your code here


## Exercise 6: The ADF Test on the Changes
(a) Run the same ADF test on the monthly changes `dy`.
(b) Make the decision at the 1 percent level.
(c) Why is running the test twice (level, then changes) the standard routine rather than
overkill?

In [7]:
# your code here


## Exercise 7: The I(d) Classification
(a) Based on Exercises 5 and 6, classify the yield in the I(d) language and justify in one
sentence.
(b) State what I(0), I(1) and I(2) mean and which of the three dominates financial level
series.
(c) What information does differencing throw away, and which method (outlook, not exam
material) recovers a shared long-run link between two I(1) series?

In [8]:
# your code here


## Exercise 8: The Random-Walk Forecast Band
(a) Using today's yield and the sd of the monthly changes, compute the random-walk point
forecast and the 95 percent band for h = 1, 6, 12 and 24 months.
(b) Verify the handout's headline number: 1.96 x 0.25 x sqrt(12) for the 12-month band.
(c) Draw the fan chart (flat forecast plus the widening band) and give the one-sentence
humbling lesson.

In [9]:
# your code here


## Exercise 9: The Spurious Levels Regression
(a) Regress the BTC level on the S&P 500 level (with constant). Report R-squared, the slope
t-statistic and the Durbin-Watson statistic.
(b) Apply the alarm rule of thumb and name the symptom that gives the game away.
(c) Explain, with Granger and Newbold (1974) in one sentence, why the huge t-statistic is not
evidence of a relationship.

In [10]:
# your code here


## Exercise 10: The Returns Regression and the Final Verdict
(a) Compute monthly log returns of both series and rerun the regression in returns. Report
R-squared, the slope t-statistic and DW.
(b) Compare levels versus returns in one table and give the verdict on the pair.
(c) State the two cures for spurious regression and the practical workflow you will apply to
every financial series from now on.

In [11]:
# your code here
